## 1. Setup and Configuration

In [1]:
import json
import os
import random
from pathlib import Path
from typing import Dict, List, Literal

from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from supabase.client import Client, create_client
load_dotenv()

# Load reference context for grounded guidance generation
REFERENCE_CONTEXT_PATH = "../data/plantwild/evaluation/reference_context.json"
with open(REFERENCE_CONTEXT_PATH, "r") as f:
    REFERENCE_CONTEXT_LIST = json.load(f)

# Create a lookup dictionary by class name
REFERENCE_CONTEXT = {item["class"]: item for item in REFERENCE_CONTEXT_LIST}

print(f"Loaded reference context for {len(REFERENCE_CONTEXT)} classes")


Loaded reference context for 89 classes


In [2]:
# Configuration
class Config:
    SUPABASE_URL = os.getenv("SUPABASE_URL")
    SUPABASE_SERVICE_KEY = os.getenv("SUPABASE_SERVICE_KEY")
    
    # Image storage configuration
    IMAGE_STORAGE = "supabase"  # Options: "github" or "supabase"
    SUPABASE_BUCKET = "thesis-bucket"  # Your Supabase storage bucket name
    SUPABASE_FOLDER = "plantwild/evaluation/images"  # Folder path within bucket
    
    # GitHub raw URL (fallback)
    GITHUB_BASE_URL = "https://raw.githubusercontent.com/andyathsid/react-agent-engine/main/data/planwild/evaluation/images"

METADATA_PATH = "../data/plantwild/evaluation/metadata.json"
IMAGES_DIR = "../data/plantwild/evaluation/images"
OUTPUT_PATH = "../data/langsmith/vqa_identification_examples.py"

# Random seed for reproducibility
RANDOM_SEED = 42

# Initialize Supabase client for storage URLs
supabase: Client = create_client(Config.SUPABASE_URL, Config.SUPABASE_SERVICE_KEY)

def get_image_url(filename: str) -> str:
    """Generate image URL based on storage configuration."""
    if Config.IMAGE_STORAGE == "supabase":
        # Generate Supabase public URL
        path = f"{Config.SUPABASE_FOLDER}/{filename}"
        return supabase.storage.from_(Config.SUPABASE_BUCKET).get_public_url(path)
    else:
        # Use GitHub raw URL
        return f"{Config.GITHUB_BASE_URL}/{filename}"

print(f"Metadata path: {METADATA_PATH}")
print(f"Images directory: {IMAGES_DIR}")
print(f"Output path: {OUTPUT_PATH}")
print(f"Image storage: {Config.IMAGE_STORAGE}")
if Config.IMAGE_STORAGE == "supabase":
    print(f"  - Supabase bucket: {Config.SUPABASE_BUCKET}")
    print(f"  - Folder: {Config.SUPABASE_FOLDER}")
else:
    print(f"  - GitHub URL: {Config.GITHUB_BASE_URL}")

Metadata path: ../data/plantwild/evaluation/metadata.json
Images directory: ../data/plantwild/evaluation/images
Output path: ../data/langsmith/vqa_identification_examples.py
Image storage: supabase
  - Supabase bucket: thesis-bucket
  - Folder: plantwild/evaluation/images


In [3]:
test_filename = "0000_apple_black_rot.jpg"
test_path = f"plantwild/evaluation/images/{test_filename}"

url = supabase.storage.from_("thesis-bucket").get_public_url(test_path)
print(f"Test URL: {url}")

Storage endpoint URL should have a trailing slash.
Test URL: https://ttymsbsmurxtpsrvlokw.supabase.co/storage/v1/object/public/thesis-bucket/plantwild/evaluation/images/0000_apple_black_rot.jpg


## 2. Define Prompt Templates

In [4]:
# Prompt templates for Gemini to generate user queries
PROMPT_TEMPLATES = {
    "diseased_type1_vague": """You are simulating a plant owner who notices something wrong with their plant but doesn't have deep knowledge of plant diseases.

Based on this IMAGE and information:
- Plant species: {plant_species}
- Disease: {disease_name}
- Symptoms description: {caption}

Generate a single, natural user query (1-2 sentences) where you:
1. Describe the SPECIFIC symptoms YOU SEE in the image (e.g., "spots on leaves", "yellowing edges", "wilting stems") - be specific to what's visible in the image
2. Mention the plant species
3. Ask what the disease might be AND request guidance/recommendations/treatment (use varied phrasing like "what should I do?", "how do I treat this?", "any recommendations?", "what treatment do you suggest?", "how can I manage this?")

Write as a concerned plant owner would - conversational, maybe slightly worried, but not technical. Use details from the actual image.
Return ONLY the query text, nothing else.""",

    "diseased_type2_species_only": """You are simulating a plant owner who wants a quick diagnosis.

Based on this IMAGE and information:
- Plant species: {plant_species}
- Disease: {disease_name}
- Symptoms visible in image: {caption}

Generate a single, brief user query (1-2 sentences) where you:
1. State the plant species
2. Ask what disease it has AND request treatment/management advice (use varied phrasing like "what should I do?", "how to treat?", "recommended steps?", "management advice?")

Be direct and concise. The user is relying on the image to show the symptoms.
Return ONLY the query text, nothing else.""",

    "diseased_type3_minimal": """You are simulating a plant owner who is uploading an image and wants a quick answer.

Based on this IMAGE showing disease: {disease_name}

Generate a single, brief user query (10-15 words) where you:
1. Simply ask what disease is affecting the plant (don't mention species)
2. Request guidance on what to do (use natural phrasing like "and what should I do?", "how do I fix this?", "treatment recommendations?", "what can I do about it?")

Be casual and brief. Refer to what you see in the image if needed (e.g., "these spots", "this damage").
Return ONLY the query text, nothing else.""",

    "healthy_type1_condition": """You are simulating a plant owner who wants to verify their plant's health status.

Based on this IMAGE and information:
- Plant species: {plant_species}
- Caption: {caption}

Generate a single, natural user query (1-2 sentences) where you:
1. Describe the SPECIFIC condition YOU SEE in the image (e.g., "green leaves", "looks vibrant", "no visible spots")
2. Mention the plant species
3. Ask whether it's diseased or healthy AND request preventive care recommendations (use varied phrasing like "any care tips?", "how do I keep it healthy?", "prevention recommendations?", "what should I do to maintain this?")

Be natural and observational. Use details from the actual image.
Return ONLY the query text, nothing else.""",

    "healthy_type2_species_only": """You are simulating a plant owner who wants a quick health check.

Based on this IMAGE and information:
- Plant species: {plant_species}

Generate a single, brief user query (1-2 sentences) where you:
1. State the plant species
2. Ask if it's diseased or healthy/okay AND request care/prevention advice (use varied phrasing like "any care recommendations?", "how to keep it healthy?", "preventive measures?", "maintenance tips?")

Be direct and concise.
Return ONLY the query text, nothing else.""",

    "healthy_type3_minimal": """You are simulating a plant owner who is uploading an image for a health check.

Based on this IMAGE of a healthy plant.

Generate a single, brief user query (10-15 words) where you:
1. Simply ask if the plant is diseased or healthy (don't mention species)
2. Request basic care or prevention advice (use natural phrasing like "and how do I keep it this way?", "any care tips?", "prevention advice?", "what should I do?")

Be casual and brief.
Return ONLY the query text, nothing else.""",
}


In [5]:
# Reference answer templates for Gemini generation (grounded in reference_context.json)
REFERENCE_ANSWER_TEMPLATES = {
    "diseased": """Based on this IMAGE showing a diseased plant, generate a concise reference answer (2-4 sentences) that includes both diagnosis and guidance.

Disease information:
- Disease name: {disease_name}
- Pathogen type: {pathogen_type}
- Plant species: {plant_species}

IMPORTANT: You MUST incorporate the following evidence-based management guidance in your answer:
{management_context}

The answer should:
- Identify the disease by name
- Mention the pathogen type (fungal/bacterial/viral/pest)
- Include management guidance FROM THE PROVIDED CONTEXT ABOVE (do not generate new guidance)
- Be professional but accessible
- Reference visible symptoms if appropriate

Example format:
"The symptoms are consistent with {disease_name}, a {pathogen_type} disease. [Incorporate key management steps from the provided context]"

Return ONLY the answer text, nothing else.""",

    "healthy": """Based on this IMAGE showing a healthy plant, generate a concise reference answer (2-4 sentences) that confirms health and provides preventive care guidance.

Plant species: {plant_species}

IMPORTANT: You MUST incorporate the following evidence-based maintenance guidance in your answer:
{maintenance_context}

The answer should:
- Confirm the plant is healthy
- Mention absence of disease symptoms
- Include preventive care recommendations FROM THE PROVIDED CONTEXT ABOVE (do not generate new guidance)
- Be reassuring but professional
- Reference visible healthy characteristics if appropriate

Example format:
"The plant appears healthy with no visible signs of disease or significant stress. [Incorporate key maintenance recommendations from the provided context]"

Return ONLY the answer text, nothing else.""",
}

# Reference goal templates - aligned with agent's identification-focused workflow
REFERENCE_GOAL_TEMPLATES = {
    "diseased": """Based on this IMAGE and the user query, generate a concise reference goal (1-2 sentences) that describes what the agent should accomplish.

User query: {user_query}
Disease: {disease_name}

The goal should focus on the agent's core identification task AND guidance requirement:
- Perform visual triage of symptoms visible in the image
- Use appropriate tools (detection/classification) to support identification
- Produce an evidence-grounded identification of {disease_name}
- Ground the identification in visible symptoms and available evidence
- Provide management/treatment guidance grounded in retrieval (knowledge base or web search)

Example formats:
- "Identify {disease_name} from the visible symptoms in the image using visual analysis and plant disease identification tools, then provide evidence-based management recommendations."
- "Perform visual triage of the leaf symptoms, use disease identification to confirm {disease_name}, and retrieve treatment guidance from the knowledge base."
- "Analyze the visible lesions to identify {disease_name}, grounding the diagnosis in observable symptoms, and provide management recommendations using knowledge base search."
- "Use symptom-based analysis and classification tools to identify {disease_name}, then search for treatment and prevention guidance in available resources."

The goal should reflect the agent's workflow: observe first, then use tools to support/validate, produce evidence-based identification, and provide guidance grounded in retrieval.
Return ONLY the goal text, nothing else.""",

    "healthy": """Based on this IMAGE and the user query, generate a concise reference goal (1-2 sentences) that describes what the agent should accomplish.

User query: {user_query}

The goal should focus on the agent's core identification task for health verification AND preventive care guidance:
- Perform visual assessment of the plant in the image
- Use appropriate tools if needed to verify health status
- Confirm the absence of disease symptoms
- Provide evidence-based health confirmation
- Provide preventive care recommendations to maintain plant health

Example formats:
- "Verify the plant's health status by analyzing visible characteristics, confirming the absence of disease symptoms, and provide preventive care recommendations."
- "Perform visual assessment to confirm the plant is healthy with no signs of disease, and offer maintenance guidance to keep it healthy."
- "Use visual analysis and disease identification tools to verify the plant shows no symptoms of disease, then provide preventive care recommendations."
- "Assess the plant's condition, confirm it is healthy based on visible characteristics, and recommend care practices to prevent future disease issues."

The goal should reflect the agent's workflow: observe, assess, verify health status with evidence, and provide preventive guidance.
Return ONLY the goal text, nothing else.""",
}


## 3. Helper Functions

In [6]:
def load_metadata() -> Dict:
    """Load the evaluation dataset metadata."""
    with open(METADATA_PATH, "r") as f:
        return json.load(f)


def extract_plant_species(class_name: str) -> str:
    """Extract plant species from class name (e.g., 'apple black rot' -> 'apple')."""
    parts = class_name.split()
    # Handle special cases
    if "bell pepper" in class_name:
        return "bell pepper"
    return parts[0]


def determine_pathogen_type(disease_name: str) -> Literal["fungal", "bacterial", "viral", "pest"]:
    """Determine pathogen type from disease name based on common patterns."""
    disease_lower = disease_name.lower()
    
    # Viral patterns
    if any(term in disease_lower for term in ["virus", "mosaic", "curl", "yellowing", "leafroll"]):
        return "viral"
    
    # Bacterial patterns
    if any(term in disease_lower for term in ["bacterial", "canker", "wilt", "blight", "greening"]):
        # Exception: early/late blight are fungal
        if "early blight" in disease_lower or "late blight" in disease_lower:
            return "fungal"
        return "bacterial"
    
    # Pest patterns 
    if any(term in disease_lower for term in ["pest", "mite", "insect"]):
        return "pest"
    
    # Default to fungal 
    return "fungal"


def generate_reference_answer(disease_name: str, pathogen_type: str) -> str:
    """Generate a reference answer based on disease and pathogen type."""
    template = REFERENCE_ANSWER_TEMPLATES[pathogen_type]
    return template.format(
        pathogen_type=pathogen_type,
        disease_name=disease_name
    )


def group_images_by_class(metadata: Dict) -> Dict[str, List[str]]:
    """Group image filenames by their class."""
    grouped = {}
    for filename, data in metadata.items():
        class_name = data["class"]
        if class_name not in grouped:
            grouped[class_name] = []
        grouped[class_name].append(filename)
    return grouped


In [7]:
def generate_user_query_with_gemini(
    model: ChatGoogleGenerativeAI,
    template_key: str,
    image_url: str,
    plant_species: str,
    disease_name: str,
    caption: str
) -> str:
    """Use Gemini with multimodal input to generate a natural user query based on the template."""
    prompt_text = PROMPT_TEMPLATES[template_key].format(
        plant_species=plant_species,
        disease_name=disease_name,
        caption=caption
    )
    
    # Create multimodal message with image and text
    message = HumanMessage(
        content=[
            {"type": "text", "text": prompt_text},
            {"type": "image_url", "image_url": image_url},
        ]
    )
    
    response = model.invoke([message])
    return response.content.strip()


def generate_reference_answer_with_gemini(
    model: ChatGoogleGenerativeAI,
    answer_type: str,
    image_url: str,
    class_name: str,
    disease_name: str = "",
    pathogen_type: str = "",
    plant_species: str = ""
) -> str:
    """Use Gemini with multimodal input to generate a reference answer grounded in reference_context.json."""
    
    # Get grounded context from reference_context.json
    ref_context = REFERENCE_CONTEXT.get(class_name, {})
    
    if answer_type == "diseased":
        management_context = ref_context.get("management", "No specific management guidance available.")
        prompt_text = REFERENCE_ANSWER_TEMPLATES[answer_type].format(
            disease_name=disease_name,
            pathogen_type=pathogen_type,
            plant_species=plant_species,
            management_context=management_context
        )
    else:  # healthy
        maintenance_context = ref_context.get("maintenance", "No specific maintenance guidance available.")
        prompt_text = REFERENCE_ANSWER_TEMPLATES[answer_type].format(
            plant_species=plant_species,
            maintenance_context=maintenance_context
        )
    
    # Create multimodal message with image and text
    message = HumanMessage(
        content=[
            {"type": "text", "text": prompt_text},
            {"type": "image_url", "image_url": image_url},
        ]
    )
    
    response = model.invoke([message])
    return response.content.strip()


def generate_reference_goal_with_gemini(
    model: ChatGoogleGenerativeAI,
    goal_type: str,
    image_url: str,
    user_query: str,
    disease_name: str = ""
) -> str:
    """Use Gemini with multimodal input to generate a reference goal."""
    prompt_text = REFERENCE_GOAL_TEMPLATES[goal_type].format(
        user_query=user_query,
        disease_name=disease_name
    )
    
    # Create multimodal message with image and text
    message = HumanMessage(
        content=[
            {"type": "text", "text": prompt_text},
            {"type": "image_url", "image_url": image_url},
        ]
    )
    
    response = model.invoke([message])
    return response.content.strip()


## 4. Load Metadata and Initialize Model

In [8]:
# Load metadata
metadata = load_metadata()
print(f"Loaded metadata for {len(metadata)} images")

# Group by class
grouped = group_images_by_class(metadata)
diseased_classes = {k: v for k, v in grouped.items() if metadata[v[0]]["status"] == "diseased"}
healthy_classes = {k: v for k, v in grouped.items() if metadata[v[0]]["status"] == "healthy"}

print(f"Diseased classes: {len(diseased_classes)}")
print(f"Healthy classes: {len(healthy_classes)}")
print(f"Total classes: {len(grouped)}")

# Import pandas for later use
import pandas as pd

Loaded metadata for 267 images
Diseased classes: 60
Healthy classes: 29
Total classes: 89


In [9]:
# Initialize Gemini model with evaluation API key
evaluation_api_key = os.getenv("EVALUATION_GOOGLE_API_KEY")

if not evaluation_api_key:
    raise ValueError("EVALUATION_GOOGLE_API_KEY not found in .env file. Please add it to continue.")

print(f"Using evaluation API key: {evaluation_api_key[:10]}...{evaluation_api_key[-4:]}")

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7,  # Some creativity for natural variations
    max_tokens=100,   # Queries should be brief
    google_api_key=evaluation_api_key
)


Using evaluation API key: AIzaSyDckp...Y85Q


### 4.1 Test Multimodal Input

Let's verify that multimodal input (text + image) works correctly before proceeding.

In [10]:
# Test multimodal input with a simple image description
test_image_url = get_image_url("0000_apple_black_rot.jpg")

message = HumanMessage(
    content=[
        {"type": "text", "text": "Describe what you see in this image in one sentence."},
        {"type": "image_url", "image_url": test_image_url},
    ]
)

response = model.invoke([message])
print("Multimodal test successful!")
print(f"Image URL: {test_image_url}")
print(f"Model response: {response.content}")

Multimodal test successful!
Image URL: https://ttymsbsmurxtpsrvlokw.supabase.co/storage/v1/object/public/thesis-bucket/plantwild/evaluation/images/0000_apple_black_rot.jpg
Model response: A pyramid of red and yellow apples, with one noticeably rotten apple in the center, is arranged on a vibrant yellow background.


## 5. Generate Examples (Preview Mode)

Let's first test with a few examples to verify the generation quality.

In [11]:
# Test with one diseased class
test_class = "apple black rot"
test_filenames = sorted(grouped[test_class])
plant_species = extract_plant_species(test_class)
pathogen_type = determine_pathogen_type(test_class)

print(f"Testing with: {test_class}")
print(f"Plant species: {plant_species}")
print(f"Pathogen type: {pathogen_type}")
print(f"Images: {test_filenames}")

Testing with: apple black rot
Plant species: apple
Pathogen type: fungal
Images: ['0000_apple_black_rot.jpg', '0001_apple_black_rot.jpg', '0002_apple_black_rot.jpg']


In [12]:
# Type 1: Vague symptoms
caption = metadata[test_filenames[0]]["caption"]
image_url = get_image_url(test_filenames[0])
query1 = generate_user_query_with_gemini(
    model, "diseased_type1_vague", image_url, plant_species, test_class, caption
)
print("\n[Type 1 - Vague Symptoms]")
print(f"Caption: {caption}")
print(f"Generated query: {query1}")

# Generate reference answer (grounded in reference_context.json)
answer1 = generate_reference_answer_with_gemini(
    model, "diseased", image_url, test_class, test_class, pathogen_type, plant_species
)
print(f"Reference answer: {answer1}")

# Generate reference goal
goal1 = generate_reference_goal_with_gemini(
    model, "diseased", image_url, query1, test_class
)
print(f"Reference goal: {goal1}")

# Show reference context being used
ref_context = REFERENCE_CONTEXT.get(test_class, {})
print(f"\nReference context (management): {ref_context.get('management', 'N/A')}")



[Type 1 - Vague Symptoms]
Caption: Apple black rot typically appears as circular, sunken, black lesions on the fruit, surrounded by a velvety greenish-black spore mass.
Generated query: I'm worried about my apple plant; one of the apples has developed a really dark, sunken, rotten-looking spot on the fruit. What disease might this be, and what should I do to treat it?
Reference answer: The visible symptoms of severe rot and characteristic fungal fruiting bodies on the apple are indicative of apple black rot, a common fungal disease. To manage this, promptly remove and destroy all infected leaves, fruit, and mummified fruit from the tree to prevent ascospore production, and prune infected young shoots with cracked bark during winter. Practice good sanitation by removing all crop debris after harvest, and apply protective fungicides, such as copper-based options or Bordeaux mixture, from silver tip to harvest, particularly when wet periods are forecast.
Reference answer: The visible sym

In [13]:
# Type 2: Species only
caption = metadata[test_filenames[1]]["caption"]
image_url = get_image_url(test_filenames[1])
query2 = generate_user_query_with_gemini(
    model, "diseased_type2_species_only", image_url, plant_species, test_class, caption
)
print("\n[Type 2 - Species Only]")
print(f"Caption: {caption}")
print(f"Generated query: {query2}")

# Generate reference answer (grounded in reference_context.json)
answer2 = generate_reference_answer_with_gemini(
    model, "diseased", image_url, test_class, test_class, pathogen_type, plant_species
)
print(f"Reference answer: {answer2}")

# Generate reference goal
goal2 = generate_reference_goal_with_gemini(
    model, "diseased", image_url, query2, test_class
)
print(f"Reference goal: {goal2}")



[Type 2 - Species Only]
Caption: Apple black rot causes circular, sunken, black lesions with concentric rings and orange spore masses on fruit.
Generated query: My apple plant has these symptoms. What disease is this, and what management advice do you have?
Reference answer: The observed symptoms are consistent with apple black rot, a fungal disease. Effective management includes removing and destroying all infected leaves, fruit, and mummified fruit to prevent ascospore production, along with pruning infected young shoots with cracked, coarse bark during winter and practicing good sanitation by removing all crop debris after harvest. Additionally, apply protective fungicides like copper-based fungicides or Bordeaux mixture from silver tip to harvest, especially when wet periods are forecast.
Reference answer: The observed symptoms are consistent with apple black rot, a fungal disease. Effective management includes removing and destroying all infected leaves, fruit, and mummified frui

In [14]:
# Type 3: Minimal
image_url = get_image_url(test_filenames[2])
query3 = generate_user_query_with_gemini(
    model, "diseased_type3_minimal", image_url, plant_species, test_class, ""
)
print("\n[Type 3 - Minimal]")
print(f"Generated query: {query3}")

# Generate reference answer (grounded in reference_context.json)
answer3 = generate_reference_answer_with_gemini(
    model, "diseased", image_url, test_class, test_class, pathogen_type, plant_species
)
print(f"Reference answer: {answer3}")

# Generate reference goal
goal3 = generate_reference_goal_with_gemini(
    model, "diseased", image_url, query3, test_class
)
print(f"Reference goal: {goal3}")



[Type 3 - Minimal]
Generated query: What is this rot on my plant and what should I do?
Reference answer: The dark, sunken lesion visible on the apple is consistent with apple black rot, a common fungal disease. Effective management involves removing and destroying all infected leaves, fruit, and mummified fruit from the tree, along with pruning infected young shoots during winter and practicing good sanitation by removing all crop debris after harvest. Additionally, apply protective fungicides, such as copper-based products or Bordeaux mixture, from silver tip to harvest, especially when wet periods are forecast.
Reference answer: The dark, sunken lesion visible on the apple is consistent with apple black rot, a common fungal disease. Effective management involves removing and destroying all infected leaves, fruit, and mummified fruit from the tree, along with pruning infected young shoots during winter and practicing good sanitation by removing all crop debris after harvest. Addition

In [15]:
# Test healthy examples with all three types
test_healthy_class = "apple leaf"
test_healthy_filenames = sorted(grouped[test_healthy_class])
healthy_plant_species = extract_plant_species(test_healthy_class)

print(f"\n[Healthy Class Tests: {test_healthy_class}]")
print(f"Plant species: {healthy_plant_species}")
print(f"Images: {test_healthy_filenames}")

# Type 1: Condition description
caption = metadata[test_healthy_filenames[0]]["caption"]
image_url = get_image_url(test_healthy_filenames[0])
query_h1 = generate_user_query_with_gemini(
    model, "healthy_type1_condition", image_url, healthy_plant_species, "", caption
)
print("\n[Healthy Type 1 - Condition + Species]")
print(f"Caption: {caption}")
print(f"Generated query: {query_h1}")

# Generate reference answer (grounded in reference_context.json)
answer_h1 = generate_reference_answer_with_gemini(
    model, "healthy", image_url, test_healthy_class, "", "", healthy_plant_species
)
print(f"Reference answer: {answer_h1}")

goal_h1 = generate_reference_goal_with_gemini(
    model, "healthy", image_url, query_h1, ""
)
print(f"Reference goal: {goal_h1}")

# Show reference context being used
ref_context_h = REFERENCE_CONTEXT.get(test_healthy_class, {})
print(f"\nReference context (maintenance): {ref_context_h.get('maintenance', 'N/A')}")

# Type 2: Species only
image_url = get_image_url(test_healthy_filenames[1])
query_h2 = generate_user_query_with_gemini(
    model, "healthy_type2_species_only", image_url, healthy_plant_species, "", ""
)
print("\n[Healthy Type 2 - Species Only]")
print(f"Generated query: {query_h2}")

answer_h2 = generate_reference_answer_with_gemini(
    model, "healthy", image_url, test_healthy_class, "", "", healthy_plant_species
)
print(f"Reference answer: {answer_h2}")

goal_h2 = generate_reference_goal_with_gemini(
    model, "healthy", image_url, query_h2, ""
)
print(f"Reference goal: {goal_h2}")

# Type 3: Minimal
image_url = get_image_url(test_healthy_filenames[2])
query_h3 = generate_user_query_with_gemini(
    model, "healthy_type3_minimal", image_url, healthy_plant_species, "", ""
)
print("\n[Healthy Type 3 - Minimal]")
print(f"Generated query: {query_h3}")

answer_h3 = generate_reference_answer_with_gemini(
    model, "healthy", image_url, test_healthy_class, "", "", healthy_plant_species
)
print(f"Reference answer: {answer_h3}")

goal_h3 = generate_reference_goal_with_gemini(
    model, "healthy", image_url, query_h3, ""
)
print(f"Reference goal: {goal_h3}")



[Healthy Class Tests: apple leaf]
Plant species: apple
Images: ['0180_apple_leaf.jpg', '0181_apple_leaf.jpg', '0182_apple_leaf.jpg']

[Healthy Type 1 - Condition + Species]
Caption: Apple leaf appearance is typically green, oval-shaped, serrated edges, and may turn yellow or brown with disease.
Generated query: I'm confused about my apple plant; the leaves in this image appear entirely metallic gold and bronze, not green as expected, and there's also a branch with red berries. Does this unusual coloring suggest it's diseased, and what are the best prevention recommendations to keep it healthy?

[Healthy Type 1 - Condition + Species]
Caption: Apple leaf appearance is typically green, oval-shaped, serrated edges, and may turn yellow or brown with disease.
Generated query: I'm confused about my apple plant; the leaves in this image appear entirely metallic gold and bronze, not green as expected, and there's also a branch with red berries. Does this unusual coloring suggest it's diseased,

## 6. Full Dataset Generation

## 5.1 Verify Healthy Class Structure

Let's verify that healthy classes now have 3 images each.

In [16]:
# Check healthy class structure
healthy_class_summary = []
for class_name, filenames in sorted(healthy_classes.items()):
    healthy_class_summary.append({
        "class": class_name,
        "count": len(filenames),
        "filenames": ", ".join(sorted(filenames))
    })

healthy_df = pd.DataFrame(healthy_class_summary)
print(f"Total healthy classes: {len(healthy_classes)}")
print(f"\nClasses with != 3 images:")
print(healthy_df[healthy_df["count"] != 3])
print(f"\nAll healthy classes summary:")
healthy_df

Total healthy classes: 29

Classes with != 3 images:
Empty DataFrame
Columns: [class, count, filenames]
Index: []

All healthy classes summary:


,class,count,filenames
0,apple leaf,3,"0180_apple_leaf.jpg, 0181_apple_leaf.jpg, 0182..."
1,banana leaf,3,"0183_banana_leaf.jpg, 0184_banana_leaf.jpg, 01..."
2,basil leaf,3,"0186_basil_leaf.jpg, 0187_basil_leaf.jpg, 0188..."
3,bean leaf,3,"0189_bean_leaf.jpg, 0190_bean_leaf.jpg, 0191_b..."
4,blueberry leaf,3,"0192_blueberry_leaf.jpg, 0193_blueberry_leaf.j..."
5,broccoli leaf,3,"0195_broccoli_leaf.jpg, 0196_broccoli_leaf.jpg..."
6,cabbage leaf,3,"0198_cabbage_leaf.jpg, 0199_cabbage_leaf.jpg, ..."
7,cauliflower leaf,3,"0201_cauliflower_leaf.jpg, 0202_cauliflower_le..."
8,celery leaf,3,"0204_celery_leaf.jpg, 0205_celery_leaf.jpg, 02..."
9,cherry leaf,3,"0207_cherry_leaf.jpg, 0208_cherry_leaf.jpg, 02..."


In [17]:
import time
from tqdm.notebook import tqdm

def generate_vqa_examples(metadata: Dict, model: ChatGoogleGenerativeAI, seed: int = 42) -> List[Dict]:
    """Generate VQA identification examples for all images."""
    random.seed(seed)
    examples = []
    
    # Group images by class
    grouped = group_images_by_class(metadata)
    
    # Process each class with progress bar
    for class_name in tqdm(sorted(grouped.keys()), desc="Generating examples"):
        filenames = grouped[class_name]
        class_data = metadata[filenames[0]]
        status = class_data["status"]
        plant_species = extract_plant_species(class_name)
        
        # Get reference context for this class
        ref_context = REFERENCE_CONTEXT.get(class_name, {})
        reference_context_text = ref_context.get("management" if status == "diseased" else "maintenance", "")
        
        if status == "diseased":
            # For diseased classes: 3 images with different prompt types
            if len(filenames) != 3:
                print(f"Warning: {class_name} has {len(filenames)} images, expected 3. Skipping.")
                continue
            
            disease_name = class_name
            pathogen_type = determine_pathogen_type(disease_name)
            filenames = sorted(filenames)
            
            # Type 1: Vague symptoms
            caption = metadata[filenames[0]]["caption"]
            image_url = get_image_url(filenames[0])
            query = generate_user_query_with_gemini(
                model, "diseased_type1_vague", image_url, plant_species, disease_name, caption
            )
            time.sleep(0.5)  # Rate limiting
            
            answer = generate_reference_answer_with_gemini(
                model, "diseased", image_url, class_name, disease_name, pathogen_type, plant_species
            )
            time.sleep(0.5)  # Rate limiting
            
            goal = generate_reference_goal_with_gemini(
                model, "diseased", image_url, query, disease_name
            )
            time.sleep(0.5)  # Rate limiting
            
            examples.append({
                "inputs": {
                    "user_text": query,
                    "image_url": image_url,
                },
                "outputs": {
                    "reference_answer": answer,
                    "reference_goal": goal,
                    "reference_context": reference_context_text,
                    "reference_tool_calls": [
                        {"name": "plant_disease_identification", "args": {}},
                        {"name": "knowledgebase_search", "args": {}},
                    ],
                },
                "metadata": {
                    "class": class_name,
                    "plant": plant_species,
                    "pathogen_type": pathogen_type,
                    "prompt_type": "vague_symptoms",
                    "filename": filenames[0],
                }
            })
            
            # Type 2: Species only
            caption = metadata[filenames[1]]["caption"]
            image_url = get_image_url(filenames[1])
            query = generate_user_query_with_gemini(
                model, "diseased_type2_species_only", image_url, plant_species, disease_name, caption
            )
            time.sleep(0.5)  # Rate limiting
            
            answer = generate_reference_answer_with_gemini(
                model, "diseased", image_url, class_name, disease_name, pathogen_type, plant_species
            )
            time.sleep(0.5)  # Rate limiting
            
            goal = generate_reference_goal_with_gemini(
                model, "diseased", image_url, query, disease_name
            )
            time.sleep(0.5)  # Rate limiting
            
            examples.append({
                "inputs": {
                    "user_text": query,
                    "image_url": image_url,
                },
                "outputs": {
                    "reference_answer": answer,
                    "reference_goal": goal,
                    "reference_context": reference_context_text,
                    "reference_tool_calls": [
                        {"name": "plant_disease_identification", "args": {}},
                        {"name": "knowledgebase_search", "args": {}},
                    ],
                },
                "metadata": {
                    "class": class_name,
                    "plant": plant_species,
                    "pathogen_type": pathogen_type,
                    "prompt_type": "species_only",
                    "filename": filenames[1],
                }
            })
            
            # Type 3: Minimal
            image_url = get_image_url(filenames[2])
            query = generate_user_query_with_gemini(
                model, "diseased_type3_minimal", image_url, plant_species, disease_name, ""
            )
            time.sleep(0.5)  # Rate limiting
            
            answer = generate_reference_answer_with_gemini(
                model, "diseased", image_url, class_name, disease_name, pathogen_type, plant_species
            )
            time.sleep(0.5)  # Rate limiting
            
            goal = generate_reference_goal_with_gemini(
                model, "diseased", image_url, query, disease_name
            )
            time.sleep(0.5)  # Rate limiting
            
            examples.append({
                "inputs": {
                    "user_text": query,
                    "image_url": image_url,
                },
                "outputs": {
                    "reference_answer": answer,
                    "reference_goal": goal,
                    "reference_context": reference_context_text,
                    "reference_tool_calls": [
                        {"name": "plant_disease_identification", "args": {}},
                        {"name": "knowledgebase_search", "args": {}},
                    ],
                },
                "metadata": {
                    "class": class_name,
                    "plant": plant_species,
                    "pathogen_type": pathogen_type,
                    "prompt_type": "minimal",
                    "filename": filenames[2],
                }
            })
            
        else:  # healthy
            if len(filenames) != 3:
                print(f"Warning: {class_name} has {len(filenames)} images, expected 3. Skipping.")
                continue
            
            filenames = sorted(filenames)
            
            # Type 1: Condition description
            caption = metadata[filenames[0]]["caption"]
            image_url = get_image_url(filenames[0])
            query = generate_user_query_with_gemini(
                model, "healthy_type1_condition", image_url, plant_species, "", caption
            )
            time.sleep(0.5)  # Rate limiting
            
            answer = generate_reference_answer_with_gemini(
                model, "healthy", image_url, class_name, "", "", plant_species
            )
            time.sleep(0.5)  # Rate limiting
            
            goal = generate_reference_goal_with_gemini(
                model, "healthy", image_url, query, ""
            )
            time.sleep(0.5)  # Rate limiting
            
            examples.append({
                "inputs": {
                    "user_text": query,
                    "image_url": image_url,
                },
                "outputs": {
                    "reference_answer": answer,
                    "reference_goal": goal,
                    "reference_context": reference_context_text,
                    "reference_tool_calls": [
                        {"name": "plant_disease_identification", "args": {}},
                    ],
                },
                "metadata": {
                    "class": class_name,
                    "plant": plant_species,
                    "pathogen_type": "healthy",
                    "prompt_type": "condition_description",
                    "filename": filenames[0],
                }
            })
            
            # Type 2: Species only
            image_url = get_image_url(filenames[1])
            query = generate_user_query_with_gemini(
                model, "healthy_type2_species_only", image_url, plant_species, "", ""
            )
            time.sleep(0.5)  # Rate limiting
            
            answer = generate_reference_answer_with_gemini(
                model, "healthy", image_url, class_name, "", "", plant_species
            )
            time.sleep(0.5)  # Rate limiting
            
            goal = generate_reference_goal_with_gemini(
                model, "healthy", image_url, query, ""
            )
            time.sleep(0.5)  # Rate limiting
            
            examples.append({
                "inputs": {
                    "user_text": query,
                    "image_url": image_url,
                },
                "outputs": {
                    "reference_answer": answer,
                    "reference_goal": goal,
                    "reference_context": reference_context_text,
                    "reference_tool_calls": [
                        {"name": "plant_disease_identification", "args": {}},
                    ],
                },
                "metadata": {
                    "class": class_name,
                    "plant": plant_species,
                    "pathogen_type": "healthy",
                    "prompt_type": "species_only",
                    "filename": filenames[1],
                }
            })
            
            # Type 3: Minimal
            image_url = get_image_url(filenames[2])
            query = generate_user_query_with_gemini(
                model, "healthy_type3_minimal", image_url, plant_species, "", ""
            )
            time.sleep(0.5)  # Rate limiting
            
            answer = generate_reference_answer_with_gemini(
                model, "healthy", image_url, class_name, "", "", plant_species
            )
            time.sleep(0.5)  # Rate limiting
            
            goal = generate_reference_goal_with_gemini(
                model, "healthy", image_url, query, ""
            )
            time.sleep(0.5)  # Rate limiting
            
            examples.append({
                "inputs": {
                    "user_text": query,
                    "image_url": image_url,
                },
                "outputs": {
                    "reference_answer": answer,
                    "reference_goal": goal,
                    "reference_context": reference_context_text,
                    "reference_tool_calls": [
                        {"name": "plant_disease_identification", "args": {}},
                    ],
                },
                "metadata": {
                    "class": class_name,
                    "plant": plant_species,
                    "pathogen_type": "healthy",
                    "prompt_type": "minimal",
                    "filename": filenames[2],
                }
            })
    
    return examples


In [18]:
# Generate all examples
# WARNING: This will take 30-40 minutes due to API rate limiting (3 API calls per example)
# Each example requires: query generation + answer generation + goal generation
print("Starting full dataset generation...")
print("=" * 80)

examples = generate_vqa_examples(metadata, model, seed=RANDOM_SEED)

print("=" * 80)
print(f"Generated {len(examples)} examples successfully!")
print(f"Expected: {60 * 3} diseased + {29 * 3} healthy = {60*3 + 29*3} total")

Starting full dataset generation...


Generating examples:   0%|          | 0/89 [00:00<?, ?it/s]

Generated 267 examples successfully!
Expected: 180 diseased + 87 healthy = 267 total


## 7. Inspect Generated Examples

In [19]:
# Display sample examples
import pandas as pd

sample_df = pd.DataFrame([
    {
        "prompt_type": ex["metadata"]["prompt_type"],
        "class": ex["metadata"]["class"],
        "pathogen": ex["metadata"]["pathogen_type"],
        "query": ex["inputs"]["user_text"][:50] + "..." if len(ex["inputs"]["user_text"]) > 50 else ex["inputs"]["user_text"],
        "answer": ex["outputs"]["reference_answer"][:50] + "..." if len(ex["outputs"]["reference_answer"]) > 50 else ex["outputs"]["reference_answer"],
        "goal": ex["outputs"]["reference_goal"][:50] + "..." if len(ex["outputs"]["reference_goal"]) > 50 else ex["outputs"]["reference_goal"],
    }
    for ex in examples[:10]
])

print("Sample of first 10 examples:")
sample_df

Sample of first 10 examples:


,prompt_type,class,pathogen,query,answer,goal
0,vague_symptoms,apple black rot,fungal,I've noticed one of the apples on my apple pla...,The visible rotting on the apple is consistent...,Perform visual triage of the rotten apple symp...
1,species_only,apple black rot,fungal,My apple plant shows these symptoms. What dise...,The visible leaf distortions and spots are con...,Visually triage the symptoms in the image and ...
2,minimal,apple black rot,fungal,"What's this rot on my fruit, and what can I do...","The visible dark, sunken lesion on the apple i...",Analyze the visible symptoms on the apple frui...
3,condition_description,apple leaf,healthy,My apple plant leaves look entirely gold and m...,The plant appears healthy with no visible sign...,Visually assess the objects in the image to de...
4,species_only,apple leaf,healthy,"For my apple tree, does it appear healthy, and...",The apple fruits and accompanying leaves in th...,Acknowledge that the provided image of harvest...
5,minimal,apple leaf,healthy,Is this plant healthy or diseased? And how do ...,"The apple plant appears healthy, displaying vi...",Perform a visual assessment of the plant to co...
6,vague_symptoms,apple mosaic virus,viral,My apple tree leaves are developing distinct y...,The image displays characteristic mosaic patte...,Perform visual triage of the leaf symptoms in ...
7,species_only,apple mosaic virus,viral,My apple tree has these symptoms; what disease...,The visible mosaic patterns and leaf distortio...,Perform visual triage of the visible leaf symp...
8,minimal,apple mosaic virus,viral,"What's causing these yellow spots on my plant,...","The symptoms, including prominent mosaic patte...",Perform visual triage of the visible leaf symp...
9,vague_symptoms,apple rust,fungal,I just noticed this bright orange spot and som...,The visible bright orange spot on the apple le...,Perform visual triage of the leaf symptoms in ...


In [20]:
# Statistics
from collections import Counter

prompt_type_counts = Counter(ex["metadata"]["prompt_type"] for ex in examples)
pathogen_type_counts = Counter(ex["metadata"]["pathogen_type"] for ex in examples)

print("=" * 80)
print("DATASET STATISTICS")
print("=" * 80)
print(f"\nTotal examples: {len(examples)}")
print(f"\nBy prompt type:")
for pt, count in sorted(prompt_type_counts.items()):
    print(f"  - {pt}: {count}")
print(f"\nBy pathogen type:")
for path_type, count in sorted(pathogen_type_counts.items()):
    print(f"  - {path_type}: {count}")
print("=" * 80)

DATASET STATISTICS

Total examples: 267

By prompt type:
  - condition_description: 29
  - minimal: 89
  - species_only: 89
  - vague_symptoms: 60

By pathogen type:
  - bacterial: 27
  - fungal: 126
  - healthy: 87
  - viral: 27


In [21]:
# Preview how examples will be distributed across the three lists
type1_count = sum(1 for ex in examples if ex["metadata"]["prompt_type"] in ["vague_symptoms", "condition_description"])
type2_count = sum(1 for ex in examples if ex["metadata"]["prompt_type"] == "species_only")
type3_count = sum(1 for ex in examples if ex["metadata"]["prompt_type"] == "minimal")

print("=" * 80)
print("DISTRIBUTION ACROSS LISTS")
print("=" * 80)
print(f"\nTYPE1_DETAILED_EXAMPLES: {type1_count} examples")
print(f"  - Diseased (vague_symptoms): {sum(1 for ex in examples if ex['metadata']['prompt_type'] == 'vague_symptoms')}")
print(f"  - Healthy (condition_description): {sum(1 for ex in examples if ex['metadata']['prompt_type'] == 'condition_description')}")

print(f"\nTYPE2_SPECIES_EXAMPLES: {type2_count} examples")
print(f"  - Diseased + Healthy (species_only): {type2_count}")

print(f"\nTYPE3_MINIMAL_EXAMPLES: {type3_count} examples")
print(f"  - Diseased + Healthy (minimal): {type3_count}")

print(f"\nTotal: {type1_count + type2_count + type3_count} examples")
print("=" * 80)

DISTRIBUTION ACROSS LISTS

TYPE1_DETAILED_EXAMPLES: 89 examples
  - Diseased (vague_symptoms): 60
  - Healthy (condition_description): 29

TYPE2_SPECIES_EXAMPLES: 89 examples
  - Diseased + Healthy (species_only): 89

TYPE3_MINIMAL_EXAMPLES: 89 examples
  - Diseased + Healthy (minimal): 89

Total: 267 examples


### 7.1 Preview Examples by List

Let's look at sample examples from each of the three lists.

In [22]:
# Separate examples by prompt type
type1_examples = [ex for ex in examples if ex["metadata"]["prompt_type"] in ["vague_symptoms", "condition_description"]]
type2_examples = [ex for ex in examples if ex["metadata"]["prompt_type"] == "species_only"]
type3_examples = [ex for ex in examples if ex["metadata"]["prompt_type"] == "minimal"]

print("=" * 80)
print("SAMPLE FROM TYPE1_DETAILED_EXAMPLES")
print("=" * 80)
for i, ex in enumerate(type1_examples[:3]):
    meta = ex["metadata"]
    print(f"\n[Example {i+1}] {meta['class']} ({meta['pathogen_type']})")
    print(f"Query: {ex['inputs']['user_text'][:100]}...")
    print(f"Goal: {ex['outputs']['reference_goal'][:100]}...")

print("\n" + "=" * 80)
print("SAMPLE FROM TYPE2_SPECIES_EXAMPLES")
print("=" * 80)
for i, ex in enumerate(type2_examples[:3]):
    meta = ex["metadata"]
    print(f"\n[Example {i+1}] {meta['class']} ({meta['pathogen_type']})")
    print(f"Query: {ex['inputs']['user_text']}")
    print(f"Goal: {ex['outputs']['reference_goal'][:100]}...")

print("\n" + "=" * 80)
print("SAMPLE FROM TYPE3_MINIMAL_EXAMPLES")
print("=" * 80)
for i, ex in enumerate(type3_examples[:3]):
    meta = ex["metadata"]
    print(f"\n[Example {i+1}] {meta['class']} ({meta['pathogen_type']})")
    print(f"Query: {ex['inputs']['user_text']}")
    print(f"Goal: {ex['outputs']['reference_goal'][:100]}...")

SAMPLE FROM TYPE1_DETAILED_EXAMPLES

[Example 1] apple black rot (fungal)
Query: I've noticed one of the apples on my apple plant has developed a really dark, sunken, rotten spot ri...
Goal: Perform visual triage of the rotten apple symptoms visible in the image, use disease identification ...

[Example 2] apple leaf (healthy)
Query: My apple plant leaves look entirely gold and metallic in this picture, not green. Is this indicative...
Goal: Visually assess the objects in the image to determine their identity. Confirm these are artificial d...

[Example 3] apple mosaic virus (viral)
Query: My apple tree leaves are developing distinct yellow and green blotchy patterns, especially noticeabl...
Goal: Perform visual triage of the leaf symptoms in the image, using disease identification tools to confi...

SAMPLE FROM TYPE2_SPECIES_EXAMPLES

[Example 1] apple black rot (fungal)
Query: My apple plant shows these symptoms. What disease is this, and what steps should I take to manage it?
Goal: 

## 8. Save to Python File

The examples will be saved in three separate lists:

**List 1 - TYPE1_DETAILED_EXAMPLES:**
- Diseased: vague_symptoms (describe symptoms + species)
- Healthy: condition_description (describe condition + species + ask if diseased)

**List 2 - TYPE2_SPECIES_EXAMPLES:**
- Both diseased & healthy with species_only prompt type

**List 3 - TYPE3_MINIMAL_EXAMPLES:**
- Both diseased & healthy with minimal prompt type

A combined `VQA_IDENTIFICATION_EXAMPLES` list is also provided for backward compatibility.

In [28]:
def save_examples_to_python_file(examples: List[Dict], output_path: str):
    """Save examples as a Python file with three separate lists by prompt type."""
    
    # Separate examples by prompt type
    type1_examples = []  # vague_symptoms / condition_description
    type2_examples = []  # species_only
    type3_examples = []  # minimal
    
    for ex in examples:
        prompt_type = ex["metadata"]["prompt_type"]
        if prompt_type in ["vague_symptoms", "condition_description"]:
            type1_examples.append(ex)
        elif prompt_type == "species_only":
            type2_examples.append(ex)
        elif prompt_type == "minimal":
            type3_examples.append(ex)
    
    # Helper function to escape strings for Python
    def escape_string(s):
        return s.replace('\\', '\\\\').replace('"', '\\"').replace('\n', '\\n')
    
    # Create the Python file content
    content = '''"""
VQA Identification Examples for Plant Disease Identification Agent

Auto-generated using Gemini 2.5 Flash from PlantWild evaluation dataset.
- 180 diseased examples (60 classes x 3 prompt types)
- 87 healthy examples (29 classes x 3 prompt types)
- Total: 267 examples organized into 3 lists by prompt type

Generation Method:
- All user queries, reference answers, and reference goals are AI-generated
- Uses multimodal prompting (text + image) for image-specific, contextual content
- Each example required 3 API calls: query + answer + goal generation
- Reference context (management/maintenance guidance) is grounded in reference_context.json

List 1 - TYPE1_DETAILED_EXAMPLES (89 examples):
- Diseased: vague_symptoms (User describes specific symptoms + mentions species)
- Healthy: condition_description (User describes condition + mentions species + asks if diseased)

List 2 - TYPE2_SPECIES_EXAMPLES (89 examples):
- Diseased: User mentions species and asks for diagnosis
- Healthy: User mentions species + asks if diseased

List 3 - TYPE3_MINIMAL_EXAMPLES (89 examples):
- Diseased: User asks minimally without mentioning species
- Healthy: User asks if diseased without mentioning species

All examples include reference_context field with evidence-based guidance from reference_context.json.
"""

# Type 1: Detailed/descriptive queries (vague_symptoms + condition_description)
TYPE1_DETAILED_EXAMPLES = [
'''
    
    # Add Type 1 examples
    for i, example in enumerate(type1_examples):
        content += "    {\n"
        content += '        "inputs": {\n'
        content += f'            "user_text": "{escape_string(example["inputs"]["user_text"])}",\n'
        content += f'            "image_url": "{example["inputs"]["image_url"]}",\n'
        content += '        },\n'
        content += '        "outputs": {\n'
        content += f'            "reference_answer": "{escape_string(example["outputs"]["reference_answer"])}",\n'
        content += f'            "reference_goal": "{escape_string(example["outputs"]["reference_goal"])}",\n'
        content += f'            "reference_context": "{escape_string(example["outputs"]["reference_context"])}",\n'
        content += f'            "reference_tool_calls": {example["outputs"]["reference_tool_calls"]},\n'
        content += '        },\n'
        
        meta = example["metadata"]
        content += f'        # Class: {meta["class"]} | Plant: {meta["plant"]} | Type: {meta["pathogen_type"]} | Prompt: {meta["prompt_type"]}\n'
        
        if i < len(type1_examples) - 1:
            content += "    },\n"
        else:
            content += "    }\n"
    
    content += "]\n\n"
    
    # Add Type 2 examples
    content += "# Type 2: Species-only queries\n"
    content += "TYPE2_SPECIES_EXAMPLES = [\n"
    
    for i, example in enumerate(type2_examples):
        content += "    {\n"
        content += '        "inputs": {\n'
        content += f'            "user_text": "{escape_string(example["inputs"]["user_text"])}",\n'
        content += f'            "image_url": "{example["inputs"]["image_url"]}",\n'
        content += '        },\n'
        content += '        "outputs": {\n'
        content += f'            "reference_answer": "{escape_string(example["outputs"]["reference_answer"])}",\n'
        content += f'            "reference_goal": "{escape_string(example["outputs"]["reference_goal"])}",\n'
        content += f'            "reference_context": "{escape_string(example["outputs"]["reference_context"])}",\n'
        content += f'            "reference_tool_calls": {example["outputs"]["reference_tool_calls"]},\n'
        content += '        },\n'
        
        meta = example["metadata"]
        content += f'        # Class: {meta["class"]} | Plant: {meta["plant"]} | Type: {meta["pathogen_type"]} | Prompt: {meta["prompt_type"]}\n'
        
        if i < len(type2_examples) - 1:
            content += "    },\n"
        else:
            content += "    }\n"
    
    content += "]\n\n"
    
    # Add Type 3 examples
    content += "# Type 3: Minimal queries\n"
    content += "TYPE3_MINIMAL_EXAMPLES = [\n"
    
    for i, example in enumerate(type3_examples):
        content += "    {\n"
        content += '        "inputs": {\n'
        content += f'            "user_text": "{escape_string(example["inputs"]["user_text"])}",\n'
        content += f'            "image_url": "{example["inputs"]["image_url"]}",\n'
        content += '        },\n'
        content += '        "outputs": {\n'
        content += f'            "reference_answer": "{escape_string(example["outputs"]["reference_answer"])}",\n'
        content += f'            "reference_goal": "{escape_string(example["outputs"]["reference_goal"])}",\n'
        content += f'            "reference_context": "{escape_string(example["outputs"]["reference_context"])}",\n'
        content += f'            "reference_tool_calls": {example["outputs"]["reference_tool_calls"]},\n'
        content += '        },\n'
        
        meta = example["metadata"]
        content += f'        # Class: {meta["class"]} | Plant: {meta["plant"]} | Type: {meta["pathogen_type"]} | Prompt: {meta["prompt_type"]}\n'
        
        if i < len(type3_examples) - 1:
            content += "    },\n"
        else:
            content += "    }\n"
    
    content += "]\n\n"
    
    # Add combined list for backward compatibility
    content += "# Combined list (all examples) - for backward compatibility\n"
    content += "VQA_IDENTIFICATION_EXAMPLES = (\n"
    content += "    TYPE1_DETAILED_EXAMPLES + \n"
    content += "    TYPE2_SPECIES_EXAMPLES + \n"
    content += "    TYPE3_MINIMAL_EXAMPLES\n"
    content += ")\n"
    
    # Write to file
    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, "w") as f:
        f.write(content)
    
    print(f"Saved examples to {output_path}")
    print(f"  - TYPE1_DETAILED_EXAMPLES: {len(type1_examples)} examples")
    print(f"  - TYPE2_SPECIES_EXAMPLES: {len(type2_examples)} examples")
    print(f"  - TYPE3_MINIMAL_EXAMPLES: {len(type3_examples)} examples")
    print(f"  - Total: {len(examples)} examples")


In [29]:
# Save to Python file
save_examples_to_python_file(examples, OUTPUT_PATH)

print(f"Dataset saved to: {OUTPUT_PATH}")

Saved examples to ../data/langsmith/vqa_identification_examples.py
  - TYPE1_DETAILED_EXAMPLES: 89 examples
  - TYPE2_SPECIES_EXAMPLES: 89 examples
  - TYPE3_MINIMAL_EXAMPLES: 89 examples
  - Total: 267 examples
Dataset saved to: ../data/langsmith/vqa_identification_examples.py


## 9. Preview Generated File

Preview the first few examples from the saved file to verify formatting.

In [30]:
# Read and display first 80 lines of the generated file to see the structure
with open(OUTPUT_PATH, "r") as f:
    lines = f.readlines()[:80]
    preview = "".join(lines)

print("=" * 80)
print("PREVIEW OF GENERATED FILE")
print("=" * 80)
print(preview)
print(f"\n... (showing first 80 of {len(open(OUTPUT_PATH).readlines())} total lines)")
print("=" * 80)
print("\nFile contains three separate lists:")
print("  - TYPE1_DETAILED_EXAMPLES")
print("  - TYPE2_SPECIES_EXAMPLES")
print("  - TYPE3_MINIMAL_EXAMPLES")
print("  - VQA_IDENTIFICATION_EXAMPLES (combined)")
print("=" * 80)

PREVIEW OF GENERATED FILE
"""
VQA Identification Examples for Plant Disease Identification Agent

Auto-generated using Gemini 2.5 Flash from PlantWild evaluation dataset.
- 180 diseased examples (60 classes x 3 prompt types)
- 87 healthy examples (29 classes x 3 prompt types)
- Total: 267 examples organized into 3 lists by prompt type

Generation Method:
- All user queries, reference answers, and reference goals are AI-generated
- Uses multimodal prompting (text + image) for image-specific, contextual content
- Each example required 3 API calls: query + answer + goal generation
- Reference context (management/maintenance guidance) is grounded in reference_context.json

List 1 - TYPE1_DETAILED_EXAMPLES (89 examples):
- Diseased: vague_symptoms (User describes specific symptoms + mentions species)
- Healthy: condition_description (User describes condition + mentions species + asks if diseased)

List 2 - TYPE2_SPECIES_EXAMPLES (89 examples):
- Diseased: User mentions species and asks for 

## 10. Export Examples as JSON (Optional)

You can also save examples as JSON for easier inspection or alternative use.

In [31]:
# Optional: Save as JSON for easier inspection
json_output_path = OUTPUT_PATH.replace(".py", ".json")

with open(json_output_path, "w") as f:
    json.dump(examples, f, indent=2)

print(f"✓ Also saved as JSON: {json_output_path}")
print(f"  Total size: {len(json.dumps(examples))} bytes")

✓ Also saved as JSON: ../data/langsmith/vqa_identification_examples.json
  Total size: 522814 bytes


## 11. Quick Validation Test

Test importing the generated file to ensure it's valid Python.

In [32]:
# Test importing the generated file
import sys
sys.path.insert(0, os.path.abspath(".."))

try:
    from data.langsmith.vqa_identification_examples import (
        TYPE1_DETAILED_EXAMPLES,
        TYPE2_SPECIES_EXAMPLES,
        TYPE3_MINIMAL_EXAMPLES,
        VQA_IDENTIFICATION_EXAMPLES
    )
    
    print("✓ Import successful!")
    print(f"\nList breakdown:")
    print(f"  TYPE1_DETAILED_EXAMPLES: {len(TYPE1_DETAILED_EXAMPLES)} examples")
    print(f"  TYPE2_SPECIES_EXAMPLES: {len(TYPE2_SPECIES_EXAMPLES)} examples")
    print(f"  TYPE3_MINIMAL_EXAMPLES: {len(TYPE3_MINIMAL_EXAMPLES)} examples")
    print(f"  VQA_IDENTIFICATION_EXAMPLES (combined): {len(VQA_IDENTIFICATION_EXAMPLES)} examples")
    
    print(f"\nFirst example from TYPE1_DETAILED_EXAMPLES:")
    print(f"  User text: {TYPE1_DETAILED_EXAMPLES[0]['inputs']['user_text']}")
    print(f"  Reference answer: {TYPE1_DETAILED_EXAMPLES[0]['outputs']['reference_answer']}")
    print(f"  Reference goal: {TYPE1_DETAILED_EXAMPLES[0]['outputs']['reference_goal']}")
    
    print(f"\nFirst example from TYPE2_SPECIES_EXAMPLES:")
    print(f"  User text: {TYPE2_SPECIES_EXAMPLES[0]['inputs']['user_text']}")
    
    print(f"\nFirst example from TYPE3_MINIMAL_EXAMPLES:")
    print(f"  User text: {TYPE3_MINIMAL_EXAMPLES[0]['inputs']['user_text']}")
    
except Exception as e:
    print(f"❌ Import failed: {e}")
    print("Check the generated file for syntax errors.")

✓ Import successful!

List breakdown:
  TYPE1_DETAILED_EXAMPLES: 89 examples
  TYPE2_SPECIES_EXAMPLES: 89 examples
  TYPE3_MINIMAL_EXAMPLES: 89 examples
  VQA_IDENTIFICATION_EXAMPLES (combined): 267 examples

First example from TYPE1_DETAILED_EXAMPLES:
  User text: I've noticed one of the apples on my apple plant has developed a really dark, sunken, rotten spot right in the middle. What could this be, and how do I treat it?
  Reference answer: The visible rotting on the apple is consistent with apple black rot, a fungal disease. To manage this, remove all infected leaves, fruit, and mummified fruit from the tree and destroy them to prevent further spread, and prune infected young shoots during winter. Additionally, apply protective fungicides, such as copper-based fungicides or Bordeaux mixture, from silver tip to harvest, especially when wet periods are forecast.
  Reference goal: Perform visual triage of the rotten apple symptoms visible in the image, use disease identification tools